# 06_test_evidence_retriever_state

## 목적

이 노트북은 `05_test_hybrid_retrieval.ipynb`에서 검증한 hybrid retrieval을
실제 LangGraph state 흐름에 연결하기 전에 state 입출력 형태로 검증한다.

입력:
- detected_risks
- missing_disclaimers
- confirmed_product_type
- extracted_text

처리:
- state에서 retrieval query 생성
- hybrid_search 실행
- report-safe evidence 생성
- evidence_context 생성
- state에 retrieved_evidences / evidence_context / evidence_quality 저장

출력:
- updated_state
- retrieved_evidences
- evidence_context
- evidence_quality

이번 노트북에서는 아직 실제 graph/workflow.py를 수정하지 않는다.
이번 노트북에서는 아직 Streamlit UI를 수정하지 않는다.

In [1]:
# 기본 import / 경로 설정
from pathlib import Path
import os
import re
import json
import pickle
from pprint import pprint
from typing import List, Dict, Any
from collections import defaultdict

import numpy as np
import pandas as pd

from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma


CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

env_path = PROJECT_ROOT / ".env"
if env_path.exists():
    load_dotenv(env_path, override=True)

RETRIEVAL_DIR = PROJECT_ROOT / "data" / "retrieval"
CHROMA_DB_DIR = PROJECT_ROOT / "data" / "chromadb"
BM25_DIR = RETRIEVAL_DIR / "bm25_index"
DEBUG_DIR = RETRIEVAL_DIR / "debug_evidence_state"

PARENTS_PATH = RETRIEVAL_DIR / "parents.jsonl"
CHILDREN_PATH = RETRIEVAL_DIR / "children.jsonl"
BM25_PATH = BM25_DIR / "bm25.pkl"
SUMMARY_PATH = DEBUG_DIR / "evidence_retriever_state_summary.json"

DEBUG_DIR.mkdir(parents=True, exist_ok=True)

COLLECTION_NAME = "complypilot_regulations_v2"
EMBEDDING_MODEL_NAME = "text-embedding-3-small"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CHROMA_DB_DIR:", CHROMA_DB_DIR)
print("BM25_PATH:", BM25_PATH)
print("COLLECTION_NAME:", COLLECTION_NAME)
print("OPENAI_API_KEY exists:", bool(os.getenv("OPENAI_API_KEY")))

c:\Users\USER\Desktop\complypilot-jb\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT: c:\Users\USER\Desktop\complypilot-jb
CHROMA_DB_DIR: c:\Users\USER\Desktop\complypilot-jb\data\chromadb
BM25_PATH: c:\Users\USER\Desktop\complypilot-jb\data\retrieval\bm25_index\bm25.pkl
COLLECTION_NAME: complypilot_regulations_v2
OPENAI_API_KEY exists: True


In [2]:
# 기존 프로젝트 state import 시도
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

try:
    from core.state import ComplianceState
    HAS_COMPLIANCE_STATE = True
    print("[OK] core.state.ComplianceState import 성공")
except Exception as e:
    HAS_COMPLIANCE_STATE = False
    ComplianceState = Dict[str, Any]
    print("[WARN] ComplianceState import 실패. dict 기반으로 테스트합니다.")
    print("error:", e)

[OK] core.state.ComplianceState import 성공


In [3]:
# JSONL 로드 및 parent map 생성
def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    """
    JSONL 파일을 읽어 dict 리스트로 반환합니다.

    Args:
        path: JSONL 파일 경로

    Return:
        dict 리스트
    """
    rows = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))

    return rows


parents = load_jsonl(PARENTS_PATH)
children = load_jsonl(CHILDREN_PATH)

parent_map = {
    row["parent_id"]: row
    for row in parents
    if row.get("parent_id")
}

print("parents:", len(parents))
print("children:", len(children))
print("parent_map:", len(parent_map))

pprint(children[0] if children else None)

parents: 608
children: 2155
parent_map: 608
{'article_no': '제1조',
 'article_title': '목적',
 'child_id': 'financial_consumer_supervisory_regulation__article_1__child_001_ef419dd8',
 'child_index': 1,
 'child_status': 'ok',
 'child_text': '이 규정은 「금융소비자 보호에 관한 법률」 및 같은 법 시행령에서 위임하는 사항과 그 시행에 필요한\n'
               '사항을 규정함을 목적으로 한다.',
 'chunk_id': 'financial_consumer_supervisory_regulation__article_1__child_001_ef419dd8',
 'doc_code': 'financial_consumer_supervisory_regulation',
 'document_priority': 3,
 'document_type': 'supervisory_regulation',
 'effective_date': '2026.4.2.',
 'has_article_no': True,
 'has_page': True,
 'has_parent_id': True,
 'is_long_child': False,
 'is_short_child': False,
 'item_no': '',
 'keywords': [],
 'law_name': '금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402)',
 'page': 1,
 'page_end': 1,
 'page_start': 1,
 'paragraph_no': '',
 'parent_id': 'financial_consumer_supervisory_regulation__article_1',
 'parent_text': '제1조(목적) 이 규정은 「금융소비자 보호에 관한 법률」 및 같은 법 시행령에서 위임하는 

In [4]:
# BM25 / Chroma 로드
with open(BM25_PATH, "rb") as f:
    bm25_payload = pickle.load(f)

bm25 = bm25_payload["bm25"]
bm25_ids = bm25_payload["ids"]
bm25_documents = bm25_payload["documents"]
bm25_metadatas = bm25_payload["metadatas"]

embedding_model = OpenAIEmbeddings(
    model=EMBEDDING_MODEL_NAME
)

vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_model,
    persist_directory=str(CHROMA_DB_DIR),
)

raw_collection = vectorstore._collection
chroma_count = raw_collection.count()

print("BM25 documents:", len(bm25_documents))
print("Chroma count:", chroma_count)

assert len(bm25_documents) == len(children), "BM25 문서 수와 children 수가 다릅니다."
assert chroma_count == len(children), "Chroma count와 children 수가 다릅니다."

print("[OK] retrieval index 로드 완료")

BM25 documents: 2155
Chroma count: 2155
[OK] retrieval index 로드 완료


In [5]:
# 기본 유틸 함수
def split_pipe_string(value: Any) -> List[str]:
    """
    pipe(|) 문자열 metadata를 리스트로 변환합니다.

    Args:
        value: metadata 값

    Return:
        문자열 리스트
    """
    if value is None:
        return []

    if isinstance(value, list):
        return [str(x) for x in value if str(x)]

    value = str(value)

    if not value:
        return []

    return [x for x in value.split("|") if x]


def tokenize_for_bm25(text: str) -> List[str]:
    """
    BM25 검색용 토큰화를 수행합니다.

    Args:
        text: 입력 텍스트

    Return:
        토큰 리스트
    """
    text = text.lower()
    tokens = re.findall(r"[가-힣A-Za-z0-9]+", text)
    tokens = [token for token in tokens if len(token) >= 2]
    return tokens


def normalize_text(text: str) -> str:
    """
    비교용 텍스트를 정규화합니다.

    Args:
        text: 입력 텍스트

    Return:
        정규화된 텍스트
    """
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def safe_float(value: Any, default: float = 0.0) -> float:
    """
    값을 float로 안전하게 변환합니다.

    Args:
        value: 입력 값
        default: 변환 실패 시 기본값

    Return:
        float 값
    """
    try:
        return float(value)
    except Exception:
        return default

In [6]:
# Query profile 생성
RISK_QUERY_RULES = {
    "approval_misleading": {
        "triggers": ["누구나 승인", "무조건 승인", "100% 승인", "승인 보장", "누구나", "무조건"],
        "expanded_query": "승인 가능성 오인 금융상품 광고 소비자 오인 조건 누구에게나 적용 승인 보장",
        "preferred_risk_tags": ["approval_misleading", "advertising_regulation"],
        "preferred_keywords": ["승인", "광고", "오인", "조건", "보장"],
    },
    "rate_condition_missing": {
        "triggers": ["최저금리", "금리", "이자율", "우대금리"],
        "expanded_query": "금리 이자율 최저금리 조건 우대금리 광고 오인 중요사항 고지 설명의무",
        "preferred_risk_tags": ["rate_condition_missing", "advertising_regulation", "explanation_duty"],
        "preferred_keywords": ["금리", "이자율", "광고", "고지", "조건"],
    },
    "fee_missing": {
        "triggers": ["수수료", "비용", "중도상환", "연체", "부대비용"],
        "expanded_query": "수수료 비용 부대비용 중도상환수수료 연체이자 고지 설명의무 금융상품 중요사항",
        "preferred_risk_tags": ["fee_missing", "explanation_duty"],
        "preferred_keywords": ["수수료", "비용", "고지", "설명", "중도상환"],
    },
    "principal_guarantee_misleading": {
        "triggers": ["원금보장", "원금 보장", "손실없음", "손실 없음"],
        "expanded_query": "원금 손실 보장 오인 투자성 상품 광고 위험 고지 설명의무",
        "preferred_risk_tags": ["principal_guarantee_misleading", "advertising_regulation", "explanation_duty"],
        "preferred_keywords": ["원금", "손실", "보장", "위험", "광고"],
    },
    "return_misleading": {
        "triggers": ["확정수익", "고수익", "수익보장", "수익률 보장"],
        "expanded_query": "수익 수익률 확정수익 보장 오인 광고 투자 위험 고지 설명의무",
        "preferred_risk_tags": ["return_misleading", "advertising_regulation", "explanation_duty"],
        "preferred_keywords": ["수익", "수익률", "보장", "오인", "광고"],
    },
    "explanation_duty": {
        "triggers": ["설명의무", "설명 의무", "중요사항", "고지"],
        "expanded_query": "설명의무 중요사항 고지 금융상품 소비자 설명 금융상품판매업자",
        "preferred_risk_tags": ["explanation_duty"],
        "preferred_keywords": ["설명", "설명의무", "중요사항", "고지"],
    },
    "unfair_solicitation": {
        "triggers": ["부당권유", "권유", "적합성", "적정성"],
        "expanded_query": "부당권유 권유 적합성 적정성 금융소비자 보호 판매 규제",
        "preferred_risk_tags": ["unfair_solicitation"],
        "preferred_keywords": ["부당권유", "권유", "적합성", "적정성"],
    },
    "advertising_regulation": {
        "triggers": ["광고", "오인", "과장", "표시"],
        "expanded_query": "금융상품 광고 표시 오인 과장 광고 금지행위 소비자 보호",
        "preferred_risk_tags": ["advertising_regulation"],
        "preferred_keywords": ["광고", "표시", "오인", "과장"],
    },
}


def build_query_profile(query: str) -> Dict[str, Any]:
    """
    원문 질의를 retrieval query profile로 변환합니다.

    Args:
        query: 사용자 질의 또는 위험 문구

    Return:
        query profile dict
    """
    matched_risk_types = []
    expanded_parts = [query]
    preferred_risk_tags = []
    preferred_keywords = []

    for risk_type, rule in RISK_QUERY_RULES.items():
        if any(trigger in query for trigger in rule["triggers"]):
            matched_risk_types.append(risk_type)
            expanded_parts.append(rule["expanded_query"])
            preferred_risk_tags.extend(rule["preferred_risk_tags"])
            preferred_keywords.extend(rule["preferred_keywords"])

    if not matched_risk_types:
        matched_risk_types = ["general"]
        expanded_parts.append("금융상품 광고 소비자 오인 중요사항 고지 설명의무")
        preferred_risk_tags.extend(["advertising_regulation", "explanation_duty"])
        preferred_keywords.extend(tokenize_for_bm25(query))

    return {
        "original_query": query,
        "expanded_query": " ".join(expanded_parts),
        "matched_risk_types": sorted(set(matched_risk_types)),
        "preferred_risk_tags": sorted(set(preferred_risk_tags)),
        "preferred_keywords": sorted(set(preferred_keywords)),
    }


pprint(build_query_profile("누구나 승인"))

{'expanded_query': '누구나 승인 승인 가능성 오인 금융상품 광고 소비자 오인 조건 누구에게나 적용 승인 보장',
 'matched_risk_types': ['approval_misleading'],
 'original_query': '누구나 승인',
 'preferred_keywords': ['광고', '보장', '승인', '오인', '조건'],
 'preferred_risk_tags': ['advertising_regulation', 'approval_misleading']}


In [7]:
# Hybrid retrieval 함수들
def vector_search(
    query_profile: Dict[str, Any],
    top_k: int = 15,
    filter_dict: Dict[str, Any] | None = None,
) -> List[Dict[str, Any]]:
    """
    Chroma vector search를 수행합니다.

    Args:
        query_profile: query profile
        top_k: 후보 개수
        filter_dict: metadata filter

    Return:
        검색 결과 리스트
    """
    query = query_profile["expanded_query"]

    if filter_dict:
        docs_with_scores = vectorstore.similarity_search_with_relevance_scores(
            query=query,
            k=top_k,
            filter=filter_dict,
        )
    else:
        docs_with_scores = vectorstore.similarity_search_with_relevance_scores(
            query=query,
            k=top_k,
        )

    rows = []

    for rank, (doc, score) in enumerate(docs_with_scores, start=1):
        metadata = dict(doc.metadata)

        rows.append({
            "rank": rank,
            "chunk_id": metadata.get("chunk_id", ""),
            "score": float(score),
            "vector_score": float(score),
            "bm25_score": 0.0,
            "text": doc.page_content,
            "retrieval_method": "vector",
            **metadata,
        })

    return rows


def bm25_search(
    query_profile: Dict[str, Any],
    top_k: int = 15,
) -> List[Dict[str, Any]]:
    """
    BM25 keyword search를 수행합니다.
    score가 0 이하인 결과는 제거합니다.

    Args:
        query_profile: query profile
        top_k: 후보 개수

    Return:
        검색 결과 리스트
    """
    query = query_profile["expanded_query"]
    query_tokens = tokenize_for_bm25(query)

    if not query_tokens:
        return []

    scores = bm25.get_scores(query_tokens)
    ranked_indices = np.argsort(scores)[::-1]

    rows = []

    for idx in ranked_indices:
        idx = int(idx)
        score = float(scores[idx])

        if score <= 0:
            continue

        metadata = dict(bm25_metadatas[idx])

        rows.append({
            "rank": len(rows) + 1,
            "chunk_id": bm25_ids[idx],
            "score": score,
            "vector_score": 0.0,
            "bm25_score": score,
            "text": bm25_documents[idx],
            "retrieval_method": "bm25",
            **metadata,
        })

        if len(rows) >= top_k:
            break

    return rows


def reciprocal_rank_fusion(
    result_lists: List[List[Dict[str, Any]]],
    k: int = 60,
) -> List[Dict[str, Any]]:
    """
    여러 검색 결과를 RRF 방식으로 병합합니다.

    Args:
        result_lists: 검색 결과 리스트들
        k: RRF 보정 상수

    Return:
        병합된 결과 리스트
    """
    scores = {}
    items = {}
    methods = defaultdict(set)
    vector_scores = defaultdict(float)
    bm25_scores = defaultdict(float)

    for result_list in result_lists:
        for rank, item in enumerate(result_list, start=1):
            chunk_id = item.get("chunk_id", "")

            if not chunk_id:
                continue

            scores[chunk_id] = scores.get(chunk_id, 0.0) + 1.0 / (k + rank)

            if chunk_id not in items:
                items[chunk_id] = dict(item)

            methods[chunk_id].add(item.get("retrieval_method", "unknown"))
            vector_scores[chunk_id] = max(vector_scores[chunk_id], safe_float(item.get("vector_score", 0.0)))
            bm25_scores[chunk_id] = max(bm25_scores[chunk_id], safe_float(item.get("bm25_score", 0.0)))

    merged = []

    for chunk_id, item in items.items():
        row = dict(item)
        row["rrf_score"] = scores[chunk_id]
        row["vector_score"] = vector_scores[chunk_id]
        row["bm25_score"] = bm25_scores[chunk_id]
        row["retrieval_method"] = "+".join(sorted(methods[chunk_id]))
        merged.append(row)

    return sorted(merged, key=lambda x: x["rrf_score"], reverse=True)

In [8]:
# Rerank / parent expansion / hybrid_search
DOCUMENT_TYPE_PRIORITY = {
    "law": 1,
    "enforcement_decree": 2,
    "supervisory_regulation": 3,
    "guideline": 4,
    "faq_or_manual": 5,
    "unknown": 9,
}


def calculate_keyword_match_bonus(row: Dict[str, Any], preferred_keywords: List[str]) -> float:
    """
    선호 키워드 포함 여부에 따른 bonus를 계산합니다.

    Args:
        row: 검색 결과
        preferred_keywords: 선호 키워드 리스트

    Return:
        bonus 점수
    """
    text = normalize_text(row.get("text", ""))
    metadata_keywords = split_pipe_string(row.get("keywords", ""))

    matched_count = 0

    for keyword in preferred_keywords:
        keyword_norm = normalize_text(keyword)

        if keyword_norm and keyword_norm in text:
            matched_count += 1
        elif keyword in metadata_keywords:
            matched_count += 1

    return min(matched_count * 0.02, 0.12)


def calculate_risk_tag_bonus(row: Dict[str, Any], preferred_risk_tags: List[str]) -> float:
    """
    risk tag 일치 bonus를 계산합니다.

    Args:
        row: 검색 결과
        preferred_risk_tags: 선호 risk tag 리스트

    Return:
        bonus 점수
    """
    row_tags = set(split_pipe_string(row.get("risk_tags", "")))
    preferred_tags = set(preferred_risk_tags)
    matched = row_tags.intersection(preferred_tags)

    return min(len(matched) * 0.05, 0.15)


def calculate_document_priority_bonus(row: Dict[str, Any]) -> float:
    """
    법령 위계 기반 bonus를 계산합니다.

    Args:
        row: 검색 결과

    Return:
        bonus 점수
    """
    document_type = row.get("document_type", "unknown")
    priority = DOCUMENT_TYPE_PRIORITY.get(document_type, 9)

    if priority == 1:
        return 0.05
    if priority == 2:
        return 0.04
    if priority == 3:
        return 0.03
    if priority == 4:
        return 0.01

    return 0.0


def apply_deterministic_rerank(
    rows: List[Dict[str, Any]],
    query_profile: Dict[str, Any],
) -> List[Dict[str, Any]]:
    """
    RRF 결과에 rule 기반 bonus를 적용해 최종 점수를 계산합니다.

    Args:
        rows: RRF 결과
        query_profile: query profile

    Return:
        rerank 결과
    """
    reranked = []

    for row in rows:
        row = dict(row)

        keyword_bonus = calculate_keyword_match_bonus(
            row,
            query_profile.get("preferred_keywords", []),
        )
        risk_tag_bonus = calculate_risk_tag_bonus(
            row,
            query_profile.get("preferred_risk_tags", []),
        )
        document_priority_bonus = calculate_document_priority_bonus(row)
        method_bonus = 0.03 if row.get("retrieval_method") == "bm25+vector" else 0.0

        row["keyword_bonus"] = keyword_bonus
        row["risk_tag_bonus"] = risk_tag_bonus
        row["document_priority_bonus"] = document_priority_bonus
        row["method_bonus"] = method_bonus

        row["final_score"] = (
            safe_float(row.get("rrf_score", 0.0))
            + keyword_bonus
            + risk_tag_bonus
            + document_priority_bonus
            + method_bonus
        )

        reranked.append(row)

    return sorted(reranked, key=lambda x: x["final_score"], reverse=True)


def attach_parent_context(row: Dict[str, Any]) -> Dict[str, Any]:
    """
    child 검색 결과에 parent 조문 전체 문맥을 붙입니다.

    Args:
        row: 검색 결과

    Return:
        parent context가 추가된 row
    """
    row = dict(row)
    parent_id = row.get("parent_id", "")
    parent = parent_map.get(parent_id, {})

    row["parent_text"] = parent.get("text", "")
    row["parent_article_no"] = parent.get("article_no", row.get("article_no", ""))
    row["parent_article_title"] = parent.get("article_title", row.get("article_title", ""))

    return row


def dedupe_by_parent_id(rows: List[Dict[str, Any]], top_k: int = 5) -> List[Dict[str, Any]]:
    """
    parent_id 기준으로 중복 근거를 제거합니다.

    Args:
        rows: 검색 결과
        top_k: 최종 반환 개수

    Return:
        dedupe된 검색 결과
    """
    deduped = []
    seen_parent_ids = set()

    for row in rows:
        parent_id = row.get("parent_id", "")

        if parent_id and parent_id in seen_parent_ids:
            continue

        if parent_id:
            seen_parent_ids.add(parent_id)

        deduped.append(row)

        if len(deduped) >= top_k:
            break

    return deduped


def hybrid_search(
    query: str,
    vector_top_k: int = 15,
    bm25_top_k: int = 15,
    final_top_k: int = 5,
    filter_dict: Dict[str, Any] | None = None,
) -> List[Dict[str, Any]]:
    """
    query를 받아 hybrid retrieval을 수행합니다.

    Args:
        query: 검색 질의
        vector_top_k: vector 후보 수
        bm25_top_k: BM25 후보 수
        final_top_k: 최종 반환 개수
        filter_dict: metadata filter

    Return:
        최종 evidence 후보
    """
    query_profile = build_query_profile(query)

    vector_rows = vector_search(query_profile, top_k=vector_top_k, filter_dict=filter_dict)
    bm25_rows = bm25_search(query_profile, top_k=bm25_top_k)

    merged_rows = reciprocal_rank_fusion([vector_rows, bm25_rows])
    reranked_rows = apply_deterministic_rerank(merged_rows, query_profile)

    expanded_rows = [
        attach_parent_context(row)
        for row in reranked_rows
    ]

    return dedupe_by_parent_id(expanded_rows, top_k=final_top_k)


test_rows = hybrid_search("수수료 고지 설명의무", final_top_k=3)

for row in test_rows:
    print(row["law_name"], row["article_no"], row["article_title"], row["final_score"], row["retrieval_method"])

금융소비자 보호에 관한 법률 시행령(대통령령)(제36287호)(20260428) 제13조 설명의무 0.21587301587301588 bm25
금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) 제13조 설명서 0.16515151515151516 vector
금융소비자 보호에 관한 법률(법률)(제21065호)(20260102) 제19조 설명의무 0.15449275362318843 bm25


In [9]:
# Report-safe evidence 변환
def to_report_evidence(row: Dict[str, Any]) -> Dict[str, Any]:
    """
    검색 결과 row를 report/UI에 안전하게 노출할 evidence format으로 변환합니다.

    Args:
        row: 검색 결과 row

    Return:
        report-safe evidence dict
    """
    law_name = row.get("law_name", "")
    article_no = row.get("article_no", "")
    article_title = row.get("article_title", "")

    doc_title = f"{law_name} {article_no}({article_title})".strip()
    snippet = row.get("text", "").replace("\n", " ").strip()

    return {
        "doc_title": doc_title,
        "page": row.get("page", row.get("page_start", "")),
        "snippet": snippet[:500],
        "score": round(float(row.get("final_score", row.get("rrf_score", 0.0))), 5),
        "retrieval_method": row.get("retrieval_method", "hybrid"),
        "document_type": row.get("document_type", ""),
        "chunk_id": row.get("chunk_id", ""),
        "parent_id": row.get("parent_id", ""),
        "risk_tags": row.get("risk_tags", ""),
    }


def build_evidence_context(evidences: List[Dict[str, Any]]) -> str:
    """
    risk_judge 또는 report_builder가 읽기 쉬운 evidence context 문자열을 생성합니다.

    Args:
        evidences: report-safe evidence 리스트

    Return:
        evidence context 문자열
    """
    lines = []

    for idx, evidence in enumerate(evidences, start=1):
        lines.append(
            f"[근거 {idx}] {evidence.get('doc_title', '')} / page={evidence.get('page', '')}\n"
            f"- retrieval_method: {evidence.get('retrieval_method', '')}\n"
            f"- score: {evidence.get('score', '')}\n"
            f"- snippet: {evidence.get('snippet', '')}"
        )

    return "\n\n".join(lines)


sample_evidences = [to_report_evidence(row) for row in test_rows]
sample_context = build_evidence_context(sample_evidences)

pprint(sample_evidences)
print(sample_context[:1200])

[{'chunk_id': 'financial_consumer_act_enforcement_decree__article_13__child_004_2919d851',
  'doc_title': '금융소비자 보호에 관한 법률 시행령(대통령령)(제36287호)(20260428) 제13조(설명의무)',
  'document_type': 'enforcement_decree',
  'page': 9,
  'parent_id': 'financial_consumer_act_enforcement_decree__article_13',
  'retrieval_method': 'bm25',
  'risk_tags': 'explanation_duty|fee_missing|rate_condition_missing',
  'score': 0.21587,
  'snippet': '[문서명: 금융소비자 보호에 관한 법률 시행령(대통령령)(제36287호)(20260428) / 조문: '
             '제13조(설명의무) / 페이지: 9] 제13조(설명의무) ④ 법 제19조제1항제1호나목4)에서 “대통령령으로 정하는 '
             '사항”이란 다음 각 호의 사항(연계투자는 제4호만 해당 한다)을 말한다. 1. 금융소비자가 부담해야 하는 수수료 '
             '2. 계약의 해지ㆍ해제 3. 증권의 환매(還買) 및 매매 4. 「온라인투자연계금융업 및 이용자 보호에 관한 법률」 '
             '제22조제1항 각 호의 정보 5. 그 밖에 제1호부터 제4호까지의 사항에 준하는 것으로서 금융위원회가 정하여 '
             '고시하는 사항'},
 {'chunk_id': 'financial_consumer_supervisory_regulation__article_13__child_001_2c45ea57',
  'doc_title': '금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402) 제13조(설명서)',
  'docu

In [10]:
# State에서 query 추출
def normalize_detected_risk_to_query(risk: Dict[str, Any]) -> str:
    """
    detected_risk row를 retrieval query로 변환합니다.

    Args:
        risk: detected risk dict

    Return:
        검색 query
    """
    parts = []

    for key in ["risk_type", "label", "matched_text", "evidence", "reason", "description"]:
        value = risk.get(key)
        if value:
            parts.append(str(value))

    return " ".join(parts).strip()


def normalize_missing_disclaimer_to_query(disclaimer: Dict[str, Any]) -> str:
    """
    missing_disclaimer row를 retrieval query로 변환합니다.

    Args:
        disclaimer: missing disclaimer dict

    Return:
        검색 query
    """
    parts = []

    for key in ["disclaimer_type", "label", "missing_item", "reason", "description"]:
        value = disclaimer.get(key)
        if value:
            parts.append(str(value))

    return " ".join(parts).strip()


def build_retrieval_queries_from_state(state: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    ComplianceState에서 retrieval query 목록을 생성합니다.

    Args:
        state: 현재 state

    Return:
        query item 리스트
    """
    query_items = []

    detected_risks = state.get("detected_risks", []) or []
    missing_disclaimers = state.get("missing_disclaimers", []) or []
    confirmed_product_type = state.get("confirmed_product_type", "") or state.get("product_type", "")

    for idx, risk in enumerate(detected_risks, start=1):
        query = normalize_detected_risk_to_query(risk)

        if query:
            if confirmed_product_type:
                query = f"{query} {confirmed_product_type}"

            query_items.append({
                "source": "detected_risks",
                "source_index": idx,
                "query": query,
                "risk_level": risk.get("risk_level", ""),
                "risk_type": risk.get("risk_type", ""),
            })

    for idx, disclaimer in enumerate(missing_disclaimers, start=1):
        query = normalize_missing_disclaimer_to_query(disclaimer)

        if query:
            if confirmed_product_type:
                query = f"{query} {confirmed_product_type}"

            query_items.append({
                "source": "missing_disclaimers",
                "source_index": idx,
                "query": query,
                "risk_level": disclaimer.get("risk_level", ""),
                "risk_type": disclaimer.get("disclaimer_type", ""),
            })

    if not query_items:
        fallback_text = state.get("extracted_text", "")[:500]

        if fallback_text:
            query_items.append({
                "source": "fallback_extracted_text",
                "source_index": 1,
                "query": fallback_text,
                "risk_level": "",
                "risk_type": "general",
            })

    return query_items

In [11]:
# evidence_retriever node 시뮬레이션
def evidence_retriever_state_node(
    state: Dict[str, Any],
    per_query_top_k: int = 3,
    final_top_k: int = 5,
) -> Dict[str, Any]:
    """
    evidence_retriever 노드처럼 state를 입력받아 evidence 관련 필드를 업데이트합니다.

    Args:
        state: 현재 ComplianceState
        per_query_top_k: query별 검색 결과 개수
        final_top_k: 전체 evidence 최종 개수

    Return:
        업데이트된 state 일부
    """
    query_items = build_retrieval_queries_from_state(state)

    all_rows = []
    retrieval_debug = []

    for query_item in query_items:
        query = query_item["query"]
        rows = hybrid_search(query, final_top_k=per_query_top_k)

        retrieval_debug.append({
            "query_item": query_item,
            "result_count": len(rows),
            "top_docs": [
                f"{row.get('law_name')} {row.get('article_no')}({row.get('article_title')})"
                for row in rows[:3]
            ],
        })

        for row in rows:
            row = dict(row)
            row["query_source"] = query_item["source"]
            row["query_source_index"] = query_item["source_index"]
            row["query"] = query
            all_rows.append(row)

    # 전체 결과를 final_score 기준으로 재정렬
    all_rows = sorted(
        all_rows,
        key=lambda x: safe_float(x.get("final_score", 0.0)),
        reverse=True,
    )

    # parent_id 기준 전체 dedupe
    deduped_rows = dedupe_by_parent_id(all_rows, top_k=final_top_k)

    retrieved_evidences = [
        to_report_evidence(row)
        for row in deduped_rows
    ]

    evidence_context = build_evidence_context(retrieved_evidences)

    evidence_quality = {
        "query_count": len(query_items),
        "raw_evidence_count": len(all_rows),
        "deduped_evidence_count": len(retrieved_evidences),
        "has_evidence": len(retrieved_evidences) > 0,
        "top_score": retrieved_evidences[0]["score"] if retrieved_evidences else 0.0,
    }

    return {
        "retrieval_queries": query_items,
        "retrieved_evidences": retrieved_evidences,
        "evidence_context": evidence_context,
        "evidence_quality": evidence_quality,
        "retrieval_debug": retrieval_debug,
    }

In [12]:
# 샘플 state 생성
sample_state = {
    "file_name": "demo_ad_text.txt",
    "confirmed_product_type": "loan",
    "extracted_text": """
    누구나 승인 가능한 최저금리 대출!
    복잡한 조건 없이 빠르게 신청 가능합니다.
    단, 수수료 및 금리 조건에 대한 상세 고지는 생략되어 있습니다.
    """,
    "detected_risks": [
        {
            "risk_type": "approval_misleading",
            "label": "승인 가능성 오인",
            "matched_text": "누구나 승인",
            "risk_level": "High",
            "reason": "승인 가능성을 보장하는 것처럼 소비자가 오인할 수 있음",
        },
        {
            "risk_type": "rate_condition_missing",
            "label": "금리 조건 고지 누락",
            "matched_text": "최저금리",
            "risk_level": "Medium",
            "reason": "최저금리 적용 조건이 함께 제시되지 않음",
        },
    ],
    "missing_disclaimers": [
        {
            "disclaimer_type": "fee_missing",
            "label": "수수료 고지 누락",
            "missing_item": "중도상환수수료 및 부대비용",
            "risk_level": "Medium",
            "reason": "비용 관련 중요사항 안내가 부족함",
        }
    ],
}

query_items = build_retrieval_queries_from_state(sample_state)

pprint(query_items)

[{'query': 'approval_misleading 승인 가능성 오인 누구나 승인 승인 가능성을 보장하는 것처럼 소비자가 오인할 수 '
           '있음 loan',
  'risk_level': 'High',
  'risk_type': 'approval_misleading',
  'source': 'detected_risks',
  'source_index': 1},
 {'query': 'rate_condition_missing 금리 조건 고지 누락 최저금리 최저금리 적용 조건이 함께 제시되지 않음 '
           'loan',
  'risk_level': 'Medium',
  'risk_type': 'rate_condition_missing',
  'source': 'detected_risks',
  'source_index': 2},
 {'query': 'fee_missing 수수료 고지 누락 중도상환수수료 및 부대비용 비용 관련 중요사항 안내가 부족함 loan',
  'risk_level': 'Medium',
  'risk_type': 'fee_missing',
  'source': 'missing_disclaimers',
  'source_index': 1}]


In [13]:
# state node 실행
state_update = evidence_retriever_state_node(
    sample_state,
    per_query_top_k=3,
    final_top_k=5,
)

print("state_update keys:")
pprint(state_update.keys())

print("\nretrieval_queries:")
pprint(state_update["retrieval_queries"])

print("\nevidence_quality:")
pprint(state_update["evidence_quality"])

print("\nretrieved_evidences:")
pprint(state_update["retrieved_evidences"])

print("\nevidence_context preview:")
print(state_update["evidence_context"][:2000])

state_update keys:
dict_keys(['retrieval_queries', 'retrieved_evidences', 'evidence_context', 'evidence_quality', 'retrieval_debug'])

retrieval_queries:
[{'query': 'approval_misleading 승인 가능성 오인 누구나 승인 승인 가능성을 보장하는 것처럼 소비자가 오인할 수 '
           '있음 loan',
  'risk_level': 'High',
  'risk_type': 'approval_misleading',
  'source': 'detected_risks',
  'source_index': 1},
 {'query': 'rate_condition_missing 금리 조건 고지 누락 최저금리 최저금리 적용 조건이 함께 제시되지 않음 '
           'loan',
  'risk_level': 'Medium',
  'risk_type': 'rate_condition_missing',
  'source': 'detected_risks',
  'source_index': 2},
 {'query': 'fee_missing 수수료 고지 누락 중도상환수수료 및 부대비용 비용 관련 중요사항 안내가 부족함 loan',
  'risk_level': 'Medium',
  'risk_type': 'fee_missing',
  'source': 'missing_disclaimers',
  'source_index': 1}]

evidence_quality:
{'deduped_evidence_count': 5,
 'has_evidence': True,
 'query_count': 3,
 'raw_evidence_count': 9,
 'top_score': 0.33613}

retrieved_evidences:
[{'chunk_id': 'financial_consumer_act__article_22__child_004_1a61a

In [14]:
# 기존 state에 merge
updated_state = {
    **sample_state,
    **state_update,
}

print("updated_state 주요 필드")

for key in [
    "file_name",
    "confirmed_product_type",
    "detected_risks",
    "missing_disclaimers",
    "retrieval_queries",
    "retrieved_evidences",
    "evidence_context",
    "evidence_quality",
]:
    print("=" * 100)
    print(key)
    pprint(updated_state.get(key))

updated_state 주요 필드
file_name
'demo_ad_text.txt'
confirmed_product_type
'loan'
detected_risks
[{'label': '승인 가능성 오인',
  'matched_text': '누구나 승인',
  'reason': '승인 가능성을 보장하는 것처럼 소비자가 오인할 수 있음',
  'risk_level': 'High',
  'risk_type': 'approval_misleading'},
 {'label': '금리 조건 고지 누락',
  'matched_text': '최저금리',
  'reason': '최저금리 적용 조건이 함께 제시되지 않음',
  'risk_level': 'Medium',
  'risk_type': 'rate_condition_missing'}]
missing_disclaimers
[{'disclaimer_type': 'fee_missing',
  'label': '수수료 고지 누락',
  'missing_item': '중도상환수수료 및 부대비용',
  'reason': '비용 관련 중요사항 안내가 부족함',
  'risk_level': 'Medium'}]
retrieval_queries
[{'query': 'approval_misleading 승인 가능성 오인 누구나 승인 승인 가능성을 보장하는 것처럼 소비자가 오인할 수 '
           '있음 loan',
  'risk_level': 'High',
  'risk_type': 'approval_misleading',
  'source': 'detected_risks',
  'source_index': 1},
 {'query': 'rate_condition_missing 금리 조건 고지 누락 최저금리 최저금리 적용 조건이 함께 제시되지 않음 '
           'loan',
  'risk_level': 'Medium',
  'risk_type': 'rate_condition_missing',
  'source': 'd

In [15]:
# state schema 호환성 체크
required_output_fields = [
    "retrieval_queries",
    "retrieved_evidences",
    "evidence_context",
    "evidence_quality",
]

required_evidence_fields = [
    "doc_title",
    "page",
    "snippet",
    "score",
    "retrieval_method",
]

checks = {
    "has_required_output_fields": all(
        field in state_update
        for field in required_output_fields
    ),
    "retrieved_evidences_is_list": isinstance(
        state_update.get("retrieved_evidences"),
        list,
    ),
    "evidence_context_is_string": isinstance(
        state_update.get("evidence_context"),
        str,
    ),
    "evidence_quality_is_dict": isinstance(
        state_update.get("evidence_quality"),
        dict,
    ),
    "has_evidence": len(state_update.get("retrieved_evidences", [])) > 0,
    "evidence_has_required_fields": all(
        all(field in evidence for field in required_evidence_fields)
        for evidence in state_update.get("retrieved_evidences", [])
    ),
    "no_local_path_in_evidence": all(
        "c:\\" not in str(evidence).lower()
        and "/users/" not in str(evidence).lower()
        and "\\users\\" not in str(evidence).lower()
        for evidence in state_update.get("retrieved_evidences", [])
    ),
}

pprint(checks)

if all(checks.values()):
    print("[OK] evidence_retriever state output 호환성 확인 완료")
else:
    print("[WARN] 일부 state output 체크 실패")

{'evidence_context_is_string': True,
 'evidence_has_required_fields': True,
 'evidence_quality_is_dict': True,
 'has_evidence': True,
 'has_required_output_fields': True,
 'no_local_path_in_evidence': True,
 'retrieved_evidences_is_list': True}
[OK] evidence_retriever state output 호환성 확인 완료


In [16]:
# 여러 scenario 테스트
scenario_states = [
    {
        "name": "approval_and_rate",
        "state": sample_state,
    },
    {
        "name": "fee_only",
        "state": {
            "file_name": "fee_ad.txt",
            "confirmed_product_type": "loan",
            "extracted_text": "수수료와 부대비용 안내 없이 대출 조건을 광고함",
            "detected_risks": [],
            "missing_disclaimers": [
                {
                    "disclaimer_type": "fee_missing",
                    "label": "수수료 고지 누락",
                    "missing_item": "수수료 및 부대비용",
                    "risk_level": "Medium",
                }
            ],
        },
    },
    {
        "name": "fallback_text_only",
        "state": {
            "file_name": "fallback.txt",
            "confirmed_product_type": "loan",
            "extracted_text": "금융상품 광고에서 중요사항 설명이 부족하고 소비자 오인 가능성이 있음",
            "detected_risks": [],
            "missing_disclaimers": [],
        },
    },
]

scenario_results = []

for scenario in scenario_states:
    name = scenario["name"]
    state = scenario["state"]

    update = evidence_retriever_state_node(
        state,
        per_query_top_k=3,
        final_top_k=5,
    )

    scenario_results.append({
        "name": name,
        "query_count": update["evidence_quality"]["query_count"],
        "evidence_count": update["evidence_quality"]["deduped_evidence_count"],
        "has_evidence": update["evidence_quality"]["has_evidence"],
        "top_doc": (
            update["retrieved_evidences"][0]["doc_title"]
            if update["retrieved_evidences"]
            else ""
        ),
    })

df_scenarios = pd.DataFrame(scenario_results)
display(df_scenarios)

,name,query_count,evidence_count,has_evidence,top_doc
0,approval_and_rate,3,5,True,금융소비자 보호에 관한 법률(법률)(제21065호)(20260102) 제22조(금융...
1,fee_only,1,3,True,금융소비자 보호에 관한 법률(법률)(제21065호)(20260102) 제19조(설명의무)
2,fallback_text_only,1,3,True,금융소비자 보호에 관한 법률(법률)(제21065호)(20260102) 제22조(금융...


In [17]:
# report_builder 입력 형태 미리보기
def build_report_evidence_section(evidences: List[Dict[str, Any]]) -> str:
    """
    보고서에 들어갈 근거 섹션 초안을 생성합니다.

    Args:
        evidences: retrieved evidence 리스트

    Return:
        보고서용 근거 섹션 문자열
    """
    if not evidences:
        return "관련 규정 근거를 찾지 못했습니다."

    lines = ["## 관련 규정 근거"]

    for idx, evidence in enumerate(evidences, start=1):
        lines.append(
            f"{idx}. {evidence.get('doc_title', '')}\n"
            f"   - page: {evidence.get('page', '')}\n"
            f"   - method: {evidence.get('retrieval_method', '')}\n"
            f"   - snippet: {evidence.get('snippet', '')}"
        )

    return "\n".join(lines)


report_evidence_section = build_report_evidence_section(
    updated_state["retrieved_evidences"]
)

print(report_evidence_section)

## 관련 규정 근거
1. 금융소비자 보호에 관한 법률(법률)(제21065호)(20260102) 제22조(금융상품등에 관한 광고 관련 준수사항)
   - page: 10
   - method: vector
   - snippet: [문서명: 금융소비자 보호에 관한 법률(법률)(제21065호)(20260102) / 조문: 제22조(금융상품등에 관한 광고 관련 준수사항) / 페이지: 10] 제22조(금융상품등에 관한 광고 관련 준수사항) ④ 금융상품판매업자등이 금융상품등에 관한 광고를 하는 경우 다음 각 호의 구분에 따른 행위를 해서는 아니 된다. 1. 보장성 상품 가. 보장한도, 보장 제한 조건, 면책사항 또는 감액지급 사항 등을 빠뜨리거나 충분히 고지하지 아니하여 제한 없 이 보장을 받을 수 있는 것으로 오인하게 하는 행위 나. 보험금이 큰 특정 내용만을 강조하거나 고액 보장 사례 등을 소개하여 보장내용이 큰 것으로 오인하게 하는 행위 다. 보험료를 일(日) 단위로 표시하거나 보험료의 산출기준을 불충분하게 설명하는 등 보험료등이 저렴한 것으로 오인하게 하는 행위 라. 만기 시 자동갱신되는 보장성 상품의 경우 갱신 시 보험료등이 인상될 수 있음을 금융소비자가 인지할 수 있 도록 충분히 고지하지 아니하는 행위 마
2. 은행업감독규정 (금융위원회고시)(제2026-10호)(20260401) 제89조(금융거래조건의 공시 및 설명 등)
   - page: 50
   - method: bm25+vector
   - snippet: [문서명: 은행업감독규정 (금융위원회고시)(제2026-10호)(20260401) / 조문: 제89조(금융거래조건의 공시 및 설명 등) / 페이지: 50] 제89조(금융거래조건의 공시 및 설명 등) ⑤ 은행은 영 제24조의5제2항제2호 각 목 외의 부분 본문에 따라 상품의 중요내용을 설명함에 있어서 은행이용 자의 합리적인 판단 또는 해당 상품의 가치에 중대한 영향을 미칠 수 있는 사항(이하 "중요사항"이라 한다)을 거 짓 또는 왜곡(불확실한 사항에

In [18]:
# summary 저장
summary = {
    "collection_name": COLLECTION_NAME,
    "chroma_count": chroma_count,
    "bm25_documents_count": len(bm25_documents),
    "sample_state_file": sample_state.get("file_name"),
    "state_update_keys": list(state_update.keys()),
    "evidence_quality": state_update.get("evidence_quality", {}),
    "scenario_results": scenario_results,
    "retrieved_evidence_count": len(state_update.get("retrieved_evidences", [])),
}

with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("summary 저장:", SUMMARY_PATH)
pprint(summary)

summary 저장: c:\Users\USER\Desktop\complypilot-jb\data\retrieval\debug_evidence_state\evidence_retriever_state_summary.json
{'bm25_documents_count': 2155,
 'chroma_count': 2155,
 'collection_name': 'complypilot_regulations_v2',
 'evidence_quality': {'deduped_evidence_count': 5,
                      'has_evidence': True,
                      'query_count': 3,
                      'raw_evidence_count': 9,
                      'top_score': 0.33613},
 'retrieved_evidence_count': 5,
 'sample_state_file': 'demo_ad_text.txt',
 'scenario_results': [{'evidence_count': 5,
                       'has_evidence': True,
                       'name': 'approval_and_rate',
                       'query_count': 3,
                       'top_doc': '금융소비자 보호에 관한 법률(법률)(제21065호)(20260102) '
                                  '제22조(금융상품등에 관한 광고 관련 준수사항)'},
                      {'evidence_count': 3,
                       'has_evidence': True,
                       'name': 'fee_only',
                 

In [19]:
# 최종 체크
print("=" * 100)
print("06_test_evidence_retriever_state 최종 체크")
print("=" * 100)

final_checks = {
    "chroma_loaded": chroma_count > 0,
    "bm25_loaded": len(bm25_documents) > 0,
    "parent_map_loaded": len(parent_map) > 0,
    "state_update_has_evidence": len(state_update.get("retrieved_evidences", [])) > 0,
    "state_update_has_context": bool(state_update.get("evidence_context", "")),
    "state_update_has_quality": isinstance(state_update.get("evidence_quality"), dict),
    "scenario_all_has_evidence": all(row["has_evidence"] for row in scenario_results),
    "report_section_generated": "관련 규정 근거" in report_evidence_section,
    "no_local_path_in_report_section": (
        "c:\\" not in report_evidence_section.lower()
        and "/users/" not in report_evidence_section.lower()
        and "\\users\\" not in report_evidence_section.lower()
    ),
}

pprint(final_checks)

if all(final_checks.values()):
    print("[OK] evidence_retriever state 연결 테스트 완료")
else:
    print("[WARN] 일부 체크가 실패했습니다. state field 또는 query builder 보완이 필요합니다.")

06_test_evidence_retriever_state 최종 체크
{'bm25_loaded': True,
 'chroma_loaded': True,
 'no_local_path_in_report_section': True,
 'parent_map_loaded': True,
 'report_section_generated': True,
 'scenario_all_has_evidence': True,
 'state_update_has_context': True,
 'state_update_has_evidence': True,
 'state_update_has_quality': True}
[OK] evidence_retriever state 연결 테스트 완료
